In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import zipfile

zip_path = "../data/favorita-grocery-sales-forecasting.zip"

with zipfile.ZipFile(zip_path, "r") as zip_file:
    print(zip_file.namelist()[:30])

In [ ]:
# Load the prepared continuous daily series.
prepared_df = pd.read_csv(
    "../data/favorita/store3_item589403_tft_ready.csv"
)

prepared_df["date"] = pd.to_datetime(
    prepared_df["date"]
)


# Load the original observed series.
raw_df = pd.read_csv(
    "../data/favorita/store3_item589403.csv"
)

raw_df["date"] = pd.to_datetime(
    raw_df["date"]
)


# Mark whether each prepared date actually existed
# in the original observed dataset.
prepared_df["was_observed"] = (
    prepared_df["date"].isin(raw_df["date"])
)


print("Prepared rows:", len(prepared_df))
print("Original observed rows:", len(raw_df))

print(
    "Observed dates:",
    prepared_df["was_observed"].sum()
)

print(
    "Inserted/missing dates:",
    (~prepared_df["was_observed"]).sum()
)

In [ ]:
# Keep only the dates that were inserted during preprocessing.
missing_dates_df = prepared_df[
    ~prepared_df["was_observed"]
].copy()


# Sort them just to make inspection easier.
missing_dates_df = missing_dates_df.sort_values("date")


print("First 20 inserted dates:")
print(
    missing_dates_df[
        ["date", "unit_sales"]
    ].head(20)
)


print("\nLast 20 inserted dates:")
print(
    missing_dates_df[
        ["date", "unit_sales"]
    ].tail(20)
)

In [ ]:
# ---------------------------------------------------------
# Find consecutive runs of inserted/missing dates.
# ---------------------------------------------------------

missing_dates_df = missing_dates_df.sort_values("date").copy()

# A new gap starts whenever the difference from the previous
# missing date is more than 1 day.
missing_dates_df["new_gap"] = (
    missing_dates_df["date"].diff().dt.days != 1
)

# Assign an ID to each consecutive missing-date run.
missing_dates_df["gap_id"] = (
    missing_dates_df["new_gap"].cumsum()
)

# Summarize each gap.
gap_summary = (
    missing_dates_df
    .groupby("gap_id")
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        gap_length_days=("date", "size")
    )
    .sort_values(
        "gap_length_days",
        ascending=False
    )
)

print("Number of separate missing-date gaps:", len(gap_summary))
print("\nLargest missing-date gaps:")
print(gap_summary.head(15))

In [ ]:
import os
import zipfile
import py7zr
import pandas as pd

zip_path = "../data/favorita-grocery-sales-forecasting.zip"

temp_dir = "../data/temp_favorita_audit"
os.makedirs(temp_dir, exist_ok=True)


# Step 1:
# Extract transactions.csv.7z from the outer Kaggle ZIP.
with zipfile.ZipFile(zip_path, "r") as outer_zip:
    outer_zip.extract(
        "transactions.csv.7z",
        path=temp_dir
    )


transactions_7z_path = os.path.join(
    temp_dir,
    "transactions.csv.7z"
)


# Step 2:
# Extract transactions.csv from the .7z archive.
with py7zr.SevenZipFile(
    transactions_7z_path,
    mode="r"
) as archive:
    archive.extractall(path=temp_dir)


# Step 3:
# Load the extracted transactions.csv file.
transactions_csv_path = os.path.join(
    temp_dir,
    "transactions.csv"
)

transactions_df = pd.read_csv(
    transactions_csv_path
)

transactions_df["date"] = pd.to_datetime(
    transactions_df["date"]
)


# Step 4:
# Check whether Store 3 was active during the two
# suspicious missing-date ranges.
audit_df = transactions_df[
    (transactions_df["store_nbr"] == 3)
    &
    (
        (
            (transactions_df["date"] >= "2016-12-10")
            & (transactions_df["date"] <= "2016-12-28")
        )
        |
        (
            (transactions_df["date"] >= "2017-06-02")
            & (transactions_df["date"] <= "2017-06-15")
        )
    )
].copy()


audit_df

In [ ]:
# =========================================================
# FINAL ZERO-IMPUTATION AUDIT
# =========================================================
#
# Check whether Store 3 was active on ALL 94 dates that were
# missing for Item 589403 and later filled with zero sales.
#
# If Store 3 has transactions on those dates, then the missing
# rows are item-level gaps rather than whole-store/data outages.

# Keep Store 3 transaction history only.
store3_transactions = transactions_df[
    transactions_df["store_nbr"] == 3
][["date", "transactions"]].copy()


# Join the 94 inserted item dates with Store 3 activity.
missing_date_audit = missing_dates_df[
    ["date"]
].merge(
    store3_transactions,
    on="date",
    how="left"
)


print("Inserted item dates:", len(missing_date_audit))

print(
    "Dates with Store 3 transaction records:",
    missing_date_audit["transactions"].notna().sum()
)

print(
    "Dates with NO Store 3 transaction record:",
    missing_date_audit["transactions"].isna().sum()
)

print(
    "Dates where Store 3 had zero transactions:",
    (missing_date_audit["transactions"] == 0).sum()
)

print(
    "Minimum Store 3 transactions on an inserted item date:",
    missing_date_audit["transactions"].min()
)

In [ ]:
# Show the 8 inserted item dates where Store 3
# itself had no transaction record.

store_level_missing_dates = missing_date_audit[
    missing_date_audit["transactions"].isna()
].copy()

store_level_missing_dates

Missing store-item dates were filled with zero sales. Store-level transaction data showed that 86 of the 94 inserted dates occurred while Store 3 was actively recording thousands of transactions, indicating item-level gaps rather than store-wide outages. The remaining 8 dates were Christmas Day or New Year's Day, when Store 3 had no transaction record. A remaining limitation is that item-level missingness cannot distinguish genuine zero demand from stockout-driven zero sales.

In [ ]:
# ---------------------------------------------------------
# Check demand volume and intermittency.
# ---------------------------------------------------------

observed_sales = raw_df["unit_sales"]

mean_daily_sales = observed_sales.mean()
median_daily_sales = observed_sales.median()

zero_sales_days = (observed_sales == 0).sum()
zero_sales_fraction = zero_sales_days / len(observed_sales)

print("Mean observed daily sales:", mean_daily_sales)
print("Median observed daily sales:", median_daily_sales)
print("Observed zero-sales days:", zero_sales_days)
print("Fraction of observed days with zero sales:",
      zero_sales_fraction)

In [ ]:
# ---------------------------------------------------------
# Verify that rolling means are built only from past sales.
# ---------------------------------------------------------

# Recreate the same shifted historical series used
# in the LightGBM feature engineering.
historical_sales = prepared_df["unit_sales"].shift(1)

# Recreate the 7-day rolling mean.
expected_rolling_mean_7 = (
    historical_sales
    .rolling(7)
    .mean()
)

# Build the rolling mean independently one more time
# to verify the logic.
manual_check = (
    prepared_df["unit_sales"]
    .shift(1)
    .rolling(7)
    .mean()
)

matches = np.isclose(
    expected_rolling_mean_7,
    manual_check,
    equal_nan=True
)

print(
    "Rolling mean uses only previous days:",
    matches.all()
)

In [ ]:
# =========================================================
# Rolling-Origin Backtesting
# =========================================================
#
# We will evaluate both TFT and LightGBM from several
# historical forecast starting points.
#
# Each origin will forecast the next 60 days.

backtest_origins = [
    pd.Timestamp("2016-12-19"),
    pd.Timestamp("2017-02-17"),
    pd.Timestamp("2017-04-18"),
    pd.Timestamp("2017-06-17"),
]


print("Backtest origins:")

for origin in backtest_origins:
    print(origin.date())

In [ ]:
# =========================================================
# Build the time boundaries for each backtest fold.
# =========================================================
#
# For every forecast origin:
#
# Training:
#   everything up to 60 days before the forecast origin
#
# Validation:
#   the 60 days immediately before the forecast origin
#
# Test:
#   the 60 days starting from the forecast origin

backtest_folds = []

for origin in backtest_origins:

    # Test period starts exactly at the forecast origin.
    test_start = origin
    test_end = origin + pd.Timedelta(days=59)

    # Validation is the 60 days immediately before test.
    validation_end = origin - pd.Timedelta(days=1)
    validation_start = origin - pd.Timedelta(days=60)

    # Training ends immediately before validation begins.
    train_end = validation_start - pd.Timedelta(days=1)

    backtest_folds.append({
        "origin": origin,
        "train_end": train_end,
        "validation_start": validation_start,
        "validation_end": validation_end,
        "test_start": test_start,
        "test_end": test_end,
    })


backtest_folds_df = pd.DataFrame(backtest_folds)

backtest_folds_df

In [ ]:
# =========================================================
# Check imputed/missing-date exposure in each backtest fold.
# =========================================================
#
# This lets us see whether one fold contains many more
# zero-filled dates than another, which could help explain
# unusual model performance later.

fold_data_quality_rows = []

for fold_number, fold in enumerate(backtest_folds, start=1):

    # Validation period
    validation_mask = (
        (prepared_df["date"] >= fold["validation_start"])
        & (prepared_df["date"] <= fold["validation_end"])
    )

    validation_df = prepared_df[
        validation_mask
    ]

    # Test period
    test_mask = (
        (prepared_df["date"] >= fold["test_start"])
        & (prepared_df["date"] <= fold["test_end"])
    )

    test_df_fold = prepared_df[
        test_mask
    ]

    # Count dates that were inserted during preprocessing.
    validation_imputed = (
        ~validation_df["was_observed"]
    ).sum()

    test_imputed = (
        ~test_df_fold["was_observed"]
    ).sum()

    fold_data_quality_rows.append({
        "fold": fold_number,
        "origin": fold["origin"],
        "validation_rows": len(validation_df),
        "validation_imputed_dates": validation_imputed,
        "test_rows": len(test_df_fold),
        "test_imputed_dates": test_imputed,
    })


fold_data_quality_df = pd.DataFrame(
    fold_data_quality_rows
)

fold_data_quality_df

In [ ]:
# =========================================================
# LightGBM feature builder for rolling backtesting
# =========================================================

def create_lightgbm_features(df):
    """
    Create the historical lag and rolling features used by LightGBM.

    Important:
    All rolling features use shift(1), so today's sales are
    never used to predict today's target.
    """

    feature_df = df.copy()

    # Historical lag features
    feature_df["lag_1"] = feature_df["unit_sales"].shift(1)
    feature_df["lag_7"] = feature_df["unit_sales"].shift(7)
    feature_df["lag_14"] = feature_df["unit_sales"].shift(14)
    feature_df["lag_28"] = feature_df["unit_sales"].shift(28)
    feature_df["lag_56"] = feature_df["unit_sales"].shift(56)

    # Only historical sales are used for rolling averages.
    historical_sales = feature_df["unit_sales"].shift(1)

    feature_df["rolling_mean_7"] = (
        historical_sales.rolling(7).mean()
    )

    feature_df["rolling_mean_14"] = (
        historical_sales.rolling(14).mean()
    )

    feature_df["rolling_mean_28"] = (
        historical_sales.rolling(28).mean()
    )

    feature_df["rolling_mean_56"] = (
        historical_sales.rolling(56).mean()
    )

    return feature_df


print("LightGBM feature function created.")

In [ ]:
# =========================================================
# LightGBM backtesting configuration
# =========================================================

lightgbm_feature_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_56",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",
    "time_idx",
    "day_of_week",
    "month",
    "promotion_status",
]

lightgbm_categorical_columns = [
    "day_of_week",
    "month",
    "promotion_status",
]

lightgbm_target_column = "unit_sales"

print("LightGBM features:", len(lightgbm_feature_columns))
print("Categorical columns:", lightgbm_categorical_columns)

In [ ]:
from lightgbm import LGBMRegressor, early_stopping


def run_lightgbm_backtest_fold(
    full_df,
    fold,
    feature_columns,
    categorical_columns
):
    """
    Train and evaluate LightGBM for one rolling backtest fold.

    The model:
    1. trains only on rows before the validation window,
    2. uses the 60-day validation window for early stopping,
    3. recursively forecasts the 60-day test window,
    4. never uses actual future test sales when building lags.
    """

    # -----------------------------------------------------
    # 1. Build lag and rolling features for the full series.
    # -----------------------------------------------------
    feature_df = create_lightgbm_features(full_df)

    # Remove early rows that do not yet have enough history
    # for lag_56 / rolling_mean_56.
    feature_df = feature_df.dropna(
        subset=feature_columns + ["unit_sales"]
    ).copy()


    # -----------------------------------------------------
    # 2. Split this fold into train / validation / test.
    # -----------------------------------------------------
    train_df = feature_df[
        feature_df["date"] <= fold["train_end"]
    ].copy()

    validation_df = feature_df[
        (feature_df["date"] >= fold["validation_start"])
        & (feature_df["date"] <= fold["validation_end"])
    ].copy()

    test_df = feature_df[
        (feature_df["date"] >= fold["test_start"])
        & (feature_df["date"] <= fold["test_end"])
    ].copy()


    # -----------------------------------------------------
    # 3. Build X/y for training and validation.
    # -----------------------------------------------------
    X_train = train_df[feature_columns].copy()
    y_train = train_df["unit_sales"].copy()

    X_validation = validation_df[feature_columns].copy()
    y_validation = validation_df["unit_sales"].copy()


    # Tell LightGBM which columns are categorical.
    for column in categorical_columns:
        X_train[column] = X_train[column].astype("category")
        X_validation[column] = X_validation[column].astype("category")


    # -----------------------------------------------------
    # 4. Train LightGBM with early stopping.
    # -----------------------------------------------------
    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_validation, y_validation)],
        eval_metric="mae",
        callbacks=[
            early_stopping(
                stopping_rounds=50,
                verbose=False
            )
        ]
    )


    # -----------------------------------------------------
    # 5. Build historical sales available at forecast origin.
    #
    # Important:
    # validation-period actual sales ARE known by the time
    # the test forecast starts, so they may be used as history.
    # -----------------------------------------------------
    history_values = full_df[
        full_df["date"] < fold["test_start"]
    ]["unit_sales"].tolist()


    predictions = []


    # -----------------------------------------------------
    # 6. Forecast the test period recursively.
    # -----------------------------------------------------
    for _, future_row in test_df.iterrows():

        prediction_row = pd.DataFrame({
            "lag_1": [history_values[-1]],
            "lag_7": [history_values[-7]],
            "lag_14": [history_values[-14]],
            "lag_28": [history_values[-28]],
            "lag_56": [history_values[-56]],

            "rolling_mean_7": [
                np.mean(history_values[-7:])
            ],
            "rolling_mean_14": [
                np.mean(history_values[-14:])
            ],
            "rolling_mean_28": [
                np.mean(history_values[-28:])
            ],
            "rolling_mean_56": [
                np.mean(history_values[-56:])
            ],

            "time_idx": [future_row["time_idx"]],
            "day_of_week": [future_row["day_of_week"]],
            "month": [future_row["month"]],
            "promotion_status": [future_row["promotion_status"]],
        })


        # Match training categories exactly.
        for column in categorical_columns:
            prediction_row[column] = pd.Categorical(
                prediction_row[column],
                categories=X_train[column].cat.categories
            )


        prediction_row = prediction_row[feature_columns]


        prediction = model.predict(
            prediction_row,
            num_iteration=model.best_iteration_
        )[0]


        # Demand cannot be negative.
        prediction = max(0.0, prediction)

        predictions.append(prediction)

        # Recursive step:
        # use our own prediction as history for later days.
        history_values.append(prediction)


    return {
        "model": model,
        "test_df": test_df,
        "predictions": np.array(predictions),
        "best_iteration": model.best_iteration_,
    }


print("LightGBM backtest function created.")

In [ ]:
fold_1_result = run_lightgbm_backtest_fold(
    prepared_df,
    backtest_folds[0],
    lightgbm_feature_columns,
    lightgbm_categorical_columns
)

print("Fold 1 predictions:", len(fold_1_result["predictions"]))
print("Best iteration:", fold_1_result["best_iteration"])

In [ ]:
# =========================================================
# Run LightGBM across ALL rolling backtest folds
# and evaluate 7 / 14 / 30 / 60-day performance.
# =========================================================

forecast_horizons = [7, 14, 30, 60]

lightgbm_backtest_results = []
lightgbm_fold_outputs = []


for fold_number, fold in enumerate(backtest_folds, start=1):

    print(f"Running LightGBM Fold {fold_number}...")

    # Train and recursively forecast this fold.
    result = run_lightgbm_backtest_fold(
        prepared_df,
        fold,
        lightgbm_feature_columns,
        lightgbm_categorical_columns
    )

    # Save full output in case we want to inspect it later.
    lightgbm_fold_outputs.append(result)

    predictions = result["predictions"]
    actual_sales = result["test_df"]["unit_sales"].to_numpy()

    # Evaluate several forecast horizons.
    for horizon in forecast_horizons:

        actual_horizon = actual_sales[:horizon]
        prediction_horizon = predictions[:horizon]

        absolute_errors = np.abs(
            actual_horizon - prediction_horizon
        )

        # Average number of units missed per day.
        mae = absolute_errors.mean()

        # Total error relative to total actual demand.
        total_actual = actual_horizon.sum()

        if total_actual != 0:
            wape = (
                absolute_errors.sum()
                / total_actual
            ) * 100
        else:
            wape = np.nan

        lightgbm_backtest_results.append({
            "fold": fold_number,
            "origin": fold["origin"],
            "horizon_days": horizon,
            "MAE": mae,
            "WAPE_percent": wape,
            "best_iteration": result["best_iteration"]
        })


# Convert results into a dataframe.
lightgbm_backtest_df = pd.DataFrame(
    lightgbm_backtest_results
)


print("\nPer-fold LightGBM results:")
display(lightgbm_backtest_df)


# =========================================================
# Aggregate across folds.
#
# Mean = average performance.
# Std  = how much performance changes between folds.
# =========================================================

lightgbm_summary_df = (
    lightgbm_backtest_df
    .groupby("horizon_days")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        WAPE_mean=("WAPE_percent", "mean"),
        WAPE_std=("WAPE_percent", "std")
    )
    .reset_index()
)



print("\nLightGBM rolling-backtest summary:")
display(lightgbm_summary_df)

### Promotion Feature Ablation Test

To determine whether promotion information contributes useful predictive
signal, LightGBM was re-evaluated across the same four rolling backtest
folds after removing `promotion_status`.

Removing the promotion feature increased mean MAE and mean WAPE across
all evaluated forecast horizons (7, 14, 30, and 60 days). Therefore,
`promotion_status` was retained as an input to the production demand
forecasting model.

This result establishes that promotion information is useful for
prediction. It does not establish a causal effect of promotions on
demand. Counterfactual promotion scenarios are therefore treated
separately from the predictive forecasting model.

In [41]:
# =========================================================
# Diagnostic 2 — LightGBM WITHOUT promotion_status
#
# Goal:
# Compare forecasting performance against our original
# LightGBM model using the EXACT SAME rolling backtest folds.
# =========================================================


# ---------------------------------------------------------
# 1. Create a feature list WITHOUT promotion_status.
# ---------------------------------------------------------

lightgbm_features_without_promotion = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_56",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_mean_56",
    "time_idx",
    "day_of_week",
    "month",
]


# promotion_status is no longer an input feature,
# so it should also be removed from the categorical columns.
categorical_without_promotion = [
    "day_of_week",
    "month",
]


# ---------------------------------------------------------
# 2. Run the SAME four rolling backtest folds.
# ---------------------------------------------------------

no_promo_feature_results = []
no_promo_feature_fold_outputs = []


for fold_number, fold in enumerate(
    backtest_folds,
    start=1
):

    print(
        f"Running Fold {fold_number} "
        "WITHOUT promotion feature..."
    )

    result = run_lightgbm_backtest_fold(
        prepared_df,
        fold,
        lightgbm_features_without_promotion,
        categorical_without_promotion
    )

    no_promo_feature_fold_outputs.append(result)

    predictions = result["predictions"]

    actual_sales = (
        result["test_df"]["unit_sales"].to_numpy()
    )


    # -----------------------------------------------------
    # Evaluate the same 7 / 14 / 30 / 60-day horizons.
    # -----------------------------------------------------

    for horizon in forecast_horizons:

        actual_horizon = actual_sales[:horizon]
        prediction_horizon = predictions[:horizon]

        absolute_errors = np.abs(
            actual_horizon - prediction_horizon
        )

        mae = absolute_errors.mean()

        total_actual = actual_horizon.sum()

        # WAPE is undefined when actual demand is zero.
        if total_actual != 0:

            wape = (
                absolute_errors.sum()
                / total_actual
            ) * 100

        else:
            wape = np.nan


        no_promo_feature_results.append({
            "fold": fold_number,
            "origin": fold["origin"],
            "horizon_days": horizon,
            "MAE": mae,
            "WAPE_percent": wape,
            "best_iteration": result["best_iteration"]
        })


# ---------------------------------------------------------
# 3. Convert per-fold results into a DataFrame.
# ---------------------------------------------------------

no_promo_feature_backtest_df = pd.DataFrame(
    no_promo_feature_results
)


# ---------------------------------------------------------
# 4. Calculate average performance across the four folds.
# ---------------------------------------------------------

no_promo_feature_summary_df = (
    no_promo_feature_backtest_df
    .groupby("horizon_days")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        WAPE_mean=("WAPE_percent", "mean"),
        WAPE_std=("WAPE_percent", "std")
    )
    .reset_index()
)


print(
    "\nLightGBM WITHOUT promotion feature:"
)

display(no_promo_feature_summary_df)

Running Fold 1 WITHOUT promotion feature...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000290 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 872
[LightGBM] [Info] Number of data points in the train set: 1331, number of used features: 12
[LightGBM] [Info] Start training from score 5.574756


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Running Fold 2 WITHOUT promotion feature...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000132 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 878
[LightGBM] [Info] Number of data points in the train set: 1391, number of used features: 12
[LightGBM] [Info] Start training from score 5.572969


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Running Fold 3 WITHOUT promotion feature...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 891
[LightGBM] [Info] Number of data points in the train set: 1451, number of used features: 12
[LightGBM] [Info] Start training from score 5.585803


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Running Fold 4 WITHOUT promotion feature...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000140 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 978
[LightGBM] [Info] Number of data points in the train set: 1511, number of used features: 12
[LightGBM] [Info] Start training from score 5.838518


c:\Users\karra\Desktop\inventory-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



LightGBM WITHOUT promotion feature:


,horizon_days,MAE_mean,MAE_std,WAPE_mean,WAPE_std
0,7,4.317224,1.426788,43.874629,10.995074
1,14,3.871035,1.400511,95.428753,108.659823
2,30,4.897338,1.788864,63.000678,29.576441
3,60,5.282723,2.141647,63.397790,17.926266


In [ ]:
# =========================================================
# TFT rolling-origin backtesting
# =========================================================

import lightning.pytorch as pl

from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss


# Keep the same TFT architecture used in the final model.
max_encoder_length = 120
max_prediction_length = 60

forecast_horizons = [7, 14, 30, 60]

tft_backtest_results = []
tft_fold_outputs = []


for fold_number, fold in enumerate(backtest_folds, start=1):

    print(f"\nRunning TFT Fold {fold_number}...")
    print(
        "Train end:", fold["train_end"].date(),
        "| Validation:", fold["validation_start"].date(),
        "to", fold["validation_end"].date(),
        "| Test:", fold["test_start"].date(),
        "to", fold["test_end"].date()
    )

    # -----------------------------------------------------
    # 1. Prepare fold-specific training data.
    # -----------------------------------------------------
    #
    # TFT gradient training only sees rows through train_end.
    fold_training_data = prepared_df[
        prepared_df["date"] <= fold["train_end"]
    ].copy()

    # Validation prediction requires historical encoder context,
    # so include all data through validation_end.
    fold_validation_data = prepared_df[
        prepared_df["date"] <= fold["validation_end"]
    ].copy()

    # Test prediction requires history through the forecast origin.
    fold_test_data = prepared_df[
        prepared_df["date"] <= fold["test_end"]
    ].copy()


    # -----------------------------------------------------
    # 2. Make sure categorical columns are strings.
    # -----------------------------------------------------
    categorical_columns_tft = [
        "series_id",
        "day_of_week",
        "month",
        "promotion_status",
    ]

    for column in categorical_columns_tft:
        fold_training_data[column] = (
            fold_training_data[column].astype(str)
        )
        fold_validation_data[column] = (
            fold_validation_data[column].astype(str)
        )
        fold_test_data[column] = (
            fold_test_data[column].astype(str)
        )


    # -----------------------------------------------------
    # 3. Create fold-specific TFT training dataset.
    # -----------------------------------------------------
    training_dataset_fold = TimeSeriesDataSet(
        fold_training_data,
        time_idx="time_idx",
        target="unit_sales",
        group_ids=["series_id"],

        min_encoder_length=max_encoder_length,
        max_encoder_length=max_encoder_length,

        min_prediction_length=max_prediction_length,
        max_prediction_length=max_prediction_length,

        static_categoricals=[
            "series_id"
        ],

        time_varying_known_categoricals=[
            "day_of_week",
            "month",
            "promotion_status",
        ],

        time_varying_known_reals=[
            "time_idx",
            "year",
            "day_of_month",
        ],

        time_varying_unknown_reals=[
            "unit_sales"
        ],

        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )


    # -----------------------------------------------------
    # 4. Create fold-specific validation dataset.
    #
    # predict=True gives us the final 60-day window ending
    # at this fold's validation_end date.
    # -----------------------------------------------------
    validation_dataset_fold = TimeSeriesDataSet.from_dataset(
        training_dataset_fold,
        fold_validation_data,
        predict=True,
        stop_randomization=True
    )


    train_dataloader_fold = (
        training_dataset_fold.to_dataloader(
            train=True,
            batch_size=64,
            shuffle=True
        )
    )

    validation_dataloader_fold = (
        validation_dataset_fold.to_dataloader(
            train=False,
            batch_size=64,
            shuffle=False
        )
    )


    # -----------------------------------------------------
    # 5. Reproducible TFT initialization.
    # -----------------------------------------------------
    pl.seed_everything(
        42,
        workers=True
    )


    tft_fold = TemporalFusionTransformer.from_dataset(
        training_dataset_fold,
        hidden_size=16,
        attention_head_size=4,
        dropout=0.1,
        hidden_continuous_size=8,
        learning_rate=0.001,
        loss=QuantileLoss(),
        log_interval=10,
        logging_metrics=None,
    )


    # -----------------------------------------------------
    # 6. Fold-specific callbacks.
    # -----------------------------------------------------
    early_stopping_fold = EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min"
    )

    checkpoint_callback_fold = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename=f"tft-backtest-fold-{fold_number}"
    )


    trainer_fold = pl.Trainer(
        max_epochs=30,
        accelerator="auto",
        devices=1,
        gradient_clip_val=0.1,
        callbacks=[
            early_stopping_fold,
            checkpoint_callback_fold
        ],
        enable_model_summary=False,
        logger=False
    )


    # -----------------------------------------------------
    # 7. Train this fold.
    # -----------------------------------------------------
    trainer_fold.fit(
        tft_fold,
        train_dataloaders=train_dataloader_fold,
        val_dataloaders=validation_dataloader_fold
    )


    # Load the best validation checkpoint.
    best_tft_fold = (
        TemporalFusionTransformer.load_from_checkpoint(
            checkpoint_callback_fold.best_model_path
        )
    )


    # -----------------------------------------------------
    # 8. Create the 60-day test dataset.
    #
    # Because fold_test_data ends exactly at test_end,
    # predict=True generates the final 60 days:
    # test_start -> test_end.
    # -----------------------------------------------------
    test_dataset_fold = TimeSeriesDataSet.from_dataset(
        training_dataset_fold,
        fold_test_data,
        predict=True,
        stop_randomization=True
    )

    test_dataloader_fold = (
        test_dataset_fold.to_dataloader(
            train=False,
            batch_size=64,
            shuffle=False
        )
    )


    # -----------------------------------------------------
    # 9. Generate the fold's TFT predictions.
    # -----------------------------------------------------
    fold_predictions_tensor = best_tft_fold.predict(
        test_dataloader_fold,
        mode="prediction"
    )

    fold_predictions = (
        fold_predictions_tensor[0]
        .detach()
        .cpu()
        .numpy()
    )


    # Actual sales for exactly this fold's test period.
    fold_actual_df = prepared_df[
        (prepared_df["date"] >= fold["test_start"])
        & (prepared_df["date"] <= fold["test_end"])
    ].copy()

    fold_actual = (
        fold_actual_df["unit_sales"]
        .to_numpy()
    )


    print(
        "Predictions:",
        len(fold_predictions),
        "| Actual values:",
        len(fold_actual)
    )


    # -----------------------------------------------------
    # 10. Evaluate 7 / 14 / 30 / 60-day horizons.
    # -----------------------------------------------------
    for horizon in forecast_horizons:

        actual_horizon = fold_actual[:horizon]
        prediction_horizon = fold_predictions[:horizon]

        absolute_errors = np.abs(
            actual_horizon - prediction_horizon
        )

        mae = absolute_errors.mean()

        total_actual = actual_horizon.sum()

        # Same zero-denominator handling used for LightGBM.
        if total_actual != 0:
            wape = (
                absolute_errors.sum()
                / total_actual
            ) * 100
        else:
            wape = np.nan


        tft_backtest_results.append({
            "fold": fold_number,
            "origin": fold["origin"],
            "horizon_days": horizon,
            "MAE": mae,
            "WAPE_percent": wape,
            "best_validation_loss":
                checkpoint_callback_fold.best_model_score.item()
        })


    # Save fold information in case we inspect it later.
    tft_fold_outputs.append({
        "fold": fold_number,
        "model": best_tft_fold,
        "predictions": fold_predictions,
        "actual": fold_actual,
        "checkpoint":
            checkpoint_callback_fold.best_model_path,
    })


# =========================================================
# Convert all fold results into one table.
# =========================================================

tft_backtest_df = pd.DataFrame(
    tft_backtest_results
)

print("\nPer-fold TFT results:")
display(tft_backtest_df)


# =========================================================
# Aggregate across folds.
# =========================================================

tft_summary_df = (
    tft_backtest_df
    .groupby("horizon_days")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        WAPE_mean=("WAPE_percent", "mean"),
        WAPE_std=("WAPE_percent", "std")
    )
    .reset_index()
)


print("\nTFT rolling-backtest summary:")
display(tft_summary_df)

In [ ]:
# =========================================================
# Fold 4 TFT sanity check
# =========================================================

fold_4_output = tft_fold_outputs[3]

fold_4_predictions = fold_4_output["predictions"]
fold_4_actual = fold_4_output["actual"]

print("Fold 4 predictions:", len(fold_4_predictions))
print("Fold 4 actual values:", len(fold_4_actual))

print("\nFirst 10 actual values:")
print(fold_4_actual[:10])

print("\nFirst 10 Fold 4 TFT predictions:")
print(fold_4_predictions[:10])

print("\nFold 4 checkpoint:")
print(fold_4_output["checkpoint"])

“TFT showed higher variance across repeated retraining/backtesting, while LightGBM was more consistent and achieved better average MAE/WAPE across all forecast horizons.”